# 1. Demo - Introduction to PlatoSim

Learning goals:
- Setting up your projects
- Running PlatoSim from a terminal
- Overview of the YAML configuration file
- Overview of PlatoSim's utilities
- Configuring PlatoSim using Python
- Running PlatoSim using Python
- Extracting information from the HDF5 output file

General introduction of architecture:

<img src="PlatoSimFlowchart.png" width="1000"/>

### Setup notebook

In [49]:
# Reload code outside notebook
%load_ext autoreload
%autoreload 2

# Configure figures in notebook
%matplotlib notebook

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Imports

In [50]:
import os
import numpy as np
import matplotlib.pyplot as plt

# PlatoSim
from platosim.matplotlibrc import setup_notebook
setup_notebook()

## Exercise 1.1 - Running PlatoSim from the terminal

As a good general habit, we start by setting up a PlatoSim project:
- <font color='orange'>If not done so before, create a general PlatoSim working directory (this will be used to store all your future PlatoSim projects)</font>
- <font color='orange'>Create a new sub-directory called ``demo0`` within your PlatoSim working directory</font>

Before running any simulations, first make sure that you have exported your path environments: 
- <font color='orange'>Open a new terminal</font>
- <font color='orange'>Activate your Conda environment</font>
- <font color='orange'>Check that the following paths are correctly set</font>
```
echo $PLATO_PROJECT_HOME
echo $PLATO_WORKDIR
echo $PYTHONPATH
```

We start getting familiar with running PlatoSim from the command line:
- <font color='orange'>Run a default simulation</font>
- <font color='orange'>Open the HDF5 output file from a terminal with ``argos <outputFile.hdf5>``</font>
- <font color='orange'>Open the log file and inspect the messages that PlatoSim provide. If you experience an error message reported by PlatoSim, this is the first file you should look at for troubleshooting.</font> 

PlatoSim is fully configured by the YAML configuration file. This file contains several **groups** that relates to different components of the PLATO payload. Let's try to change a few settings and look at the output:
- <font color='orange'>Make a copy of default YAML file either from GitHub or from your path ``$CONDA_PREFIX/inputfiles/inputfile.yaml``</font>
- <font color='orange'>Use a minute or two to get an overview of the different components of the YAML file</font>
- <font color='orange'>Say that we want to simulate an imagette, find the correct YAML block and change the dimentions of the simulated subfield. Run a new simulation and check the output</font>
- <font color='orange'>Lastly, change the number of exposures to 1 and save all possible maps to the HDF5 file. Run a new simulation and inspect the new pixel maps added to the HDF5 output file</font>

## Exercise 1.2 - General setup in Python

It's time to have a look at the Python interface that comes with PlatoSim. Configuring and setting up a simulations is done using the Python class:

In [51]:
from platosim.simulation import Simulation

You can get an overview of all the utilities of any PlatoSim class or script with the function `utilities.getFunctions()`. E.g. for the `Simulation` class this looks like:

In [52]:
from platosim.utilities import getFunctions
getFunctions(Simulation)

Simulation functions
__contains__
__getitem__
__init__
__setitem__
__str__
controlAllEffects
createDetectorTemperatureFile
createDirectory
createDriftFile
createPhotometryFile


All functions within any of PlatoSim's classes or scripts have a proper **docstring**. Always consult the docstring to get more information about a Python script. E.g. let's inspect the main function used to run PlatoSim: 

In [53]:
print(Simulation.run.__doc__)

Run the PLATO Simulator.

        Parameters
        ----------
        removeOutputFile : bool
            If the outputfile already exists before the run started, simply delete it.
        logLevel : int
            Level of verbosoty: 1 (least verbose) to 3 (most verbose)
        
        Return
        ------
        When PlatoSim fails for some reason and returns an error code (!= 0), an Exception is raised.
        


As part of the installation of PlatoSim the project directory and working directory should already have been set. Apart from their usage in these demos, these path environments are very handy when you start to making your own scripts. Thus, check that these are globally defined (i.e. the full path should appear, and if not, please first export them before continuing):

In [54]:
homeDir = os.getenv('PLATO_PROJECT_HOME')
workDir = os.getenv('PLATO_WORKDIR')
print(homeDir)
print(workDir)

/lhome/nicholas/software/PlatoSim3
/lhome/nicholas/software/workdir/


A PlatoSim simulation always starts by creating an instance of the *simulations class*, also called a **simulation object**. The simulation object is the interface to configure and setup a PlatoSim simulation. E.g. this is generally done with:
```
sim = Simulation(outputDir=outputDir, [<outputFileName>], [<inputFile>])
```
- <font color='orange'>Use the `workDir` from above to define a output directory, `outputDir` (string)</font>
- <font color='orange'>Select a `outputFileName` for your simulation (string)</font>
- <font color='orange'>By default PlatoSim uses and loads a copy of the YAML file from your installation folder. Say that we want to use the YAML file that is stored in your project folder called `demo0`, define the path to this file using the `workDir` parameter (string)</font>
- <font color='orange'>Create a simulation object `sim` using the above parameters</font>

For now we use the default YAML settings to run our simulation. Note that by default PlatoSim does not overwrite the output files from a previous simulation, if that simulation shares the same name. To avoid that PlatoSim will raise an error message, we here activate the `removeOutputFile=True` option when launching PlatoSim. Furthermore, you can trace the execution time by activating the parameter `executionTime=True`. 
- <font color='orange'>Try to run your first PlatoSim simulation from Python using `f = sim.run(removeOutputFile=True, executionTime=True)`</font>

In the directory you should now have the following files:
- `<outputFileName>.yaml`: copy of the YAML input file with the configuration parameters used (more information will follow);
- `<outputFileName>.hdf5`: resulting exposures (images, PSF, etc.);
- `<outputFileName>.log`: log file (to report actions and problems).

Note that a **simulation file** is returned when using `sim.run()`. We will return to how you can extract information from this file later.

Note that you can set the output directory afterwards as well using: 

```
sim.outputDir = outputDir
```

It is also possible to parse an absolute path to your own YAML input file using: 

```
sim.readConfigurationFile(<full/path/to/configuration/file>)
```

However, for both parameters, choose only one method since setting both will crash PlatoSim upon execution.

## Exercise 1.3 - Configure the YAML file

The simulation object allow you to make changes to any of the parameters within the YAML file. You already had a look inside the YAML file from the command line (probably using a buffer or a text editor) but you can also print the content of the YAML file using `sim.showYamlConfiguration()`. Try to do this below (don't worry, Python returns the YAML blocks in alfabetic order, hence it's normal that it doesn't look exactly like the YAML file that you inspected above):

When the simulation object is defined, changing a parameter can be done with:
```
sim["<Group>/<SubGroup>"] = <new value>
```
For example, changing the number of exposures is done with:
```
sim["ObservingParameters/NumExposures"] = 10
```

- <font color='orange'>Create a new simulation object `sim` (just use the default YAML file here)</font>
- <font color='orange'>Change the dimensions of the subfield using the simulation object as shown above. Choose a subfield of size 300 x 300 pixels (**hint:** see YAML block `SubField`)</font>
- <font color='orange'>By default the subfield zero-point is always located at the origin of CCD #1. Change the zero-point row and column of the subfield to pixel location (row, col) = (1000, 1000)</font>
- <font color='orange'>Run the simulation</font>

To gain full control over what is being saved to the HDF5 output file (without the need to open the YAML input file) you can manually set all entries in the `ControlHDF5Content` YAML block to `no` (or `False`). However, for must usecases, you are typically only interested in saving a few of these (since the more output you save to the HDF5, the more computational resources PlatoSim needs). Thus, a more efficient way is to specifically **turn off** the writing of all outputs using the function `sim.turnOffAllOutput()` (and similarly you can **turn on** the writing of all output using `sim.turnOnAllOutput()`) and then activate the writing of the entries you are interested in. 

- <font color='orange'>Using the existing simulation object, run a new simulation where you only save the pixel maps and the star positions to the HDF5 file using the above information</font>

If you are ever in doubt about which parameters that will/or are saved to the HDF5 output file, you can make a simple check using the function `sim.showAllOutput()`. Try this here:

## Exercise 1.4 - Extract information from HDF5

<img src="PlatoSimStructureOfHDF5.png" width="1000"/>

To access the HDF5 output file that contains your simulated data, you have two choices: 

  1. use the above `sim` object returned when running a simulation, or; 
  2. you can simply use the `SimFile` class which fetch a copy of the `sim` object. 

The second option is often desired since your simulation(s) potentially takes quite a while to run and you may want to fetch the simulation(s) at a later stage as well. Hence below we show how to retrieve the HDF5 output using the `SimFile` class:

In [55]:
from platosim.simfile import SimFile

Again to get an overview of the utilities of this class use:

In [56]:
getFunctions(SimFile)

SimFile functions
__del__
__init__
checkPhotometry
getApertureMask
getBackground
getBiasMapLeft
getBiasMapRight
getCoordinates
getCosmicsAffectedExposures
getCosmicsAffectedPixels


This class only takes the path to your HDF5 output file as argument:
```
f = SimFile(</path/to/outputFile.hdf5>)
```
Let's try to extract some basic simulated data products from the HDF5 file using the simfile object:
- <font color='orange'>First create a simfile object `f` as above</font>
- <font color='orange'>Inspect the docstring of the function `SimFile.getImage()` and extract the pixel maps</font>
- <font color='orange'>Inspect the docstring of the function `SimFile.showImage()` and make a plot of the simulated subfield</font>

Since the HDF5 file can contain a lot of information (just like the YAML counterpart) we here give some tips and tricks to inspect it from Python. Specifically, we have made a script called `h5` that contains a few handy functions: 

In [57]:
from platosim.h5 import h5ls, h5get

To use these functions you need to load the HDF5 file using the package `h5py`:
```
import h5py
h5file = h5py.File('<outputFile.hdf5>', "r")
```

The function **h5ls** takes an HDF5 file object or a HDF5 group as a mandatory argument and shows the complete structure of the HDF5 file or group. Each level is indicated by the following type acronyms, and for attributes their value is shown:

[G] Group <br/>
[D] Dataset <br/>
[a] Attribute

To print the entire HDF5 file is equivalent to specifying only the root group: 
```
h5.h5ls(h5file['/'])
```

- <font color='orange'>Load your HDF5 file using `h5py` as shown above</font>
- <font color='orange'>Using `h5ls`, find the CCD gain used (**hint:** see `CCD` group)</font>
- <font color='orange'>Using `h5ls`, find cosmic particle hit rate (**hint:** see `Sky` group)</font>

The prefered method to extract data from the HDF5 is using the `SimFile` class, however, you can also use **h5get** to extract data from the HDF5 file into a numpy array or python variable. This function takes two mandatory arguments, the HDF5 file object (or group) and the `<path/to/variable_or_dataset>`.

When you specify the full path, only that variable is returned. When you specify a partial path or just the name of the final dataset/attribute, the **h5get** function looks for all possible matches and returns an array with their values.

- <font color='orange'>Using `h5get` extract the CCD readout noise</font>
- <font color='orange'>Using `h5get` extract pixel images simulated</font>
- <font color='orange'>Using `h5get` extract the bias and smearing maps</font>

<font color='orange'>**Extra exercise:**</font> 

- <font color='orange'>So far we have only looked at simulating a small CCD subfield for a N-CAM. Setup a simulation for a F-CAM using the information above.</font>